# SI4006 · Proyecto final — Componente visual: la ficha llega como imagen

**Patrón visual.** El analista recibe la solicitud escaneada o fotografiada. El sistema la lee con OCR (Tesseract, en
español), la normaliza y se la pasa al clasificador de M1 (BETO + LoRA, con el pooler y el umbral de 0,54 fijados en M2),
que emite el dictamen. El texto leído se le muestra al analista para que lo revise.

**Casos con imagen.** Las 13 fichas del eval set de M2 dibujadas como documento escaneado. Son imágenes **sintéticas** y
llevan una marca visible que lo dice. Para cada una se compara el dictamen leyendo la ficha como **texto** contra
leyéndola desde la **imagen**: es la tabla de delta y también la prueba de invarianza. Además hay tres pruebas
adversariales: el mismo escaneo torcido y de baja calidad (debe dar el mismo veredicto), una foto borrosa ilegible (debe
abstenerse) y la imagen de un texto que no es una ficha (debe declararlo fuera de alcance).

**Cómo correrlo:** Colab con entorno **CPU** (no necesita GPU) → *Ejecutar todo*. Tarda unos 5 a 10 minutos.

In [ ]:
import os, sys, json, time, difflib, subprocess
from pathlib import Path
if Path('CREDIT_repo.zip').exists() and not Path('CREDIT/final/space/vision.py').exists():
    subprocess.run('rm -rf CREDIT && unzip -q -o CREDIT_repo.zip', shell=True, check=True)
if not Path('CREDIT').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/CamsMars/CREDIT'], check=True)
RAIZ = Path('CREDIT').resolve()
assert (RAIZ / 'final/space/vision.py').exists(), 'Sube CREDIT_repo.zip al panel de archivos'
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa > /dev/null
%pip install -q pytesseract peft
# Colab trae torchao 0.10, que rompe peft al cargar el adaptador LoRA de M1; el proyecto no lo usa.
!pip uninstall -y -q torchao
sys.path.insert(0, str(RAIZ / 'final' / 'space'))
import numpy as np, pandas as pd, torch
import vision as v, m1_dictamen as m
torch.manual_seed(42); np.random.seed(42)
torch.set_num_threads(os.cpu_count())
SALIDA = RAIZ / 'final' / 'salida'; IMGS = SALIDA / 'imagenes_visual'; IMGS.mkdir(parents=True, exist_ok=True)
print('Tesseract con español:', v.tesseract_disponible())

In [ ]:
eval_m2 = json.load(open(RAIZ / 'M2' / 'm2_salida' / 'eval_set.json', encoding='utf-8'))
clf = m.ClasificadorM1(device='cpu')
print(f'Clasificador de M1 cargado en {clf.segundos_carga:.0f} s · umbral {m.UMBRAL}')

def esperado(c):
    e = c['esperado'].lower()
    if e.startswith('veredicto: incumple'): return 'incumple'
    if e.startswith('veredicto: paga'): return 'paga'
    if e.startswith('fuera de alcance'): return 'fuera_de_alcance'
    if e.startswith('evidencia insuficiente'): return 'evidencia_insuficiente'
    return None   # A04: invarianza, se compara contra su ficha original

## Casos con imagen: texto contra imagen (tabla de delta e invarianza)

In [ ]:
filas = []
for c in eval_m2:
    if not m.es_ficha(c['input']):
        continue
    t0 = time.perf_counter(); d_txt = clf.dictamen(c['input']); s_txt = time.perf_counter() - t0
    img = v.ficha_a_imagen(c['input']); img.save(IMGS / f"{c['id']}.png")
    t0 = time.perf_counter(); ocr = v.leer_imagen(img); s_ocr = time.perf_counter() - t0
    d_img = clf.dictamen(ocr); s_img = time.perf_counter() - t0
    esp = esperado(c)
    filas.append({'id': c['id'], 'caso': c['etiqueta_caso'], 'esperado': esp,
                  'veredicto_texto': d_txt['veredicto'], 'veredicto_imagen': d_img['veredicto'],
                  'p_texto': None if d_txt['p'] is None else round(d_txt['p'], 4),
                  'p_imagen': None if d_img['p'] is None else round(d_img['p'], 4),
                  'acierto_texto': (d_txt['veredicto'] == esp) if esp else None,
                  'acierto_imagen': (d_img['veredicto'] == esp) if esp else None,
                  'mismo_veredicto': d_txt['veredicto'] == d_img['veredicto'],
                  'similitud_ocr': round(difflib.SequenceMatcher(None, c['input'], ocr).ratio(), 4),
                  'segundos_ocr': round(s_ocr, 2), 'segundos_total_imagen': round(s_img, 2)})
casos = pd.DataFrame(filas)
casos['delta_p'] = (casos.p_imagen - casos.p_texto).abs().round(4)
display(casos)

In [ ]:
con_esp = casos[casos.esperado.notna()]
delta = pd.DataFrame([
    {'Entrada': 'Ficha como texto (sistema de M2)', 'aciertos': f"{int(con_esp.acierto_texto.sum())}/{len(con_esp)}",
     'latencia mediana (s)': None},
    {'Entrada': 'Ficha como imagen (OCR + M1)', 'aciertos': f"{int(con_esp.acierto_imagen.sum())}/{len(con_esp)}",
     'latencia mediana (s)': round(float(casos.segundos_total_imagen.median()), 2)},
]).set_index('Entrada')
display(delta)
print(f"Invarianza: mismo veredicto en {int(casos.mismo_veredicto.sum())}/{len(casos)} fichas · "
      f"|Δp| máximo {casos.delta_p.max():.4f} · similitud OCR mínima {casos.similitud_ocr.min():.4f}")
a04 = [c for c in eval_m2 if c['id'] == 'A04']
if a04: print('A04 (perturbación de M2):', a04[0]['criterio'][:160])

## Pruebas adversariales con imagen

In [ ]:
base = eval_m2[0]
no_ficha = next((c for c in eval_m2 if not m.es_ficha(c['input'])), None)
pruebas = []
img_leve = v.degradar(v.ficha_a_imagen(base['input']), 'leve'); img_leve.save(IMGS / 'ADV_leve.png')
d_ref = clf.dictamen(base['input'])
d = clf.dictamen(v.leer_imagen(img_leve))
pruebas.append({'id': 'VIS1', 'prueba': 'escaneo torcido y de baja calidad', 'espera': f"mismo veredicto ({d_ref['veredicto']})",
                'obtiene': d['veredicto'], 'ok': d['veredicto'] == d_ref['veredicto']})
img_bor = v.degradar(v.ficha_a_imagen(base['input']), 'borrosa'); img_bor.save(IMGS / 'ADV_borrosa.png')
d = clf.dictamen(v.leer_imagen(img_bor))
pruebas.append({'id': 'VIS2', 'prueba': 'foto borrosa ilegible', 'espera': 'abstenerse (fuera_de_alcance)',
                'obtiene': d['veredicto'], 'ok': d['veredicto'] == 'fuera_de_alcance'})
if no_ficha:
    img_nf = v.ficha_a_imagen(no_ficha['input']); img_nf.save(IMGS / 'ADV_no_ficha.png')
    d = clf.dictamen(v.leer_imagen(img_nf))
    pruebas.append({'id': 'VIS3', 'prueba': f"imagen que no es una ficha ({no_ficha['id']})", 'espera': 'fuera_de_alcance',
                    'obtiene': d['veredicto'], 'ok': d['veredicto'] == 'fuera_de_alcance'})
adv = pd.DataFrame(pruebas); display(adv)

In [ ]:
casos.to_csv(SALIDA / 'visual_casos.csv', index=False, encoding='utf-8')
delta.to_csv(SALIDA / 'visual_delta.csv', encoding='utf-8')
adv.to_csv(SALIDA / 'visual_adversariales.csv', index=False, encoding='utf-8')
resumen = {'aciertos_texto': delta.loc['Ficha como texto (sistema de M2)', 'aciertos'],
           'aciertos_imagen': delta.loc['Ficha como imagen (OCR + M1)', 'aciertos'],
           'invarianza': f"{int(casos.mismo_veredicto.sum())}/{len(casos)}",
           'delta_p_max': float(casos.delta_p.max()), 'similitud_ocr_min': float(casos.similitud_ocr.min()),
           'latencia_mediana_imagen_s': float(casos.segundos_total_imagen.median()),
           'adversariales': f"{int(adv.ok.sum())}/{len(adv)}", 'umbral': m.UMBRAL,
           'tesseract': subprocess.run(['tesseract', '--version'], capture_output=True, text=True).stdout.split('\n')[0]}
json.dump(resumen, open(SALIDA / 'visual_resumen.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(json.dumps(resumen, ensure_ascii=False, indent=1))
print(); print(delta.to_markdown()); print(); print(adv.to_markdown(index=False))

In [ ]:
!cd {RAIZ} && zip -qr /content/salida_visual.zip final/salida/visual_* final/salida/imagenes_visual && echo listo
from google.colab import files
files.download('/content/salida_visual.zip')